# Tic-Tac-Toe & Connect 4 AI with MLPs (TensorFlow)

Teaching small Dense/MLP networks to play board games, two ways:

1. **Supervised learning** — a perfect minimax player labels every tic-tac-toe position, and an MLP learns to imitate it.
2. **Self-play reinforcement learning** — Deep Q-Learning with no teacher at all: the network plays thousands of games against itself and learns from wins and losses.

Experiments covered:

| # | Experiment |
|---|---|
| 1 | **Generalization** — train on half of the positions, test on positions never seen |
| 2 | **Symmetry** — use the 8 rotations/reflections of the board as data augmentation |
| 3 | **Model size** — find the smallest network that still plays well |
| 5 | **Policy + value heads** — predict the best move *and* who is winning (Functional API) |
| 6 | **Self-play** — learn tic-tac-toe with no minimax teacher |
| 7 | **Connect 4** — a game too big for full minimax (~4.5 trillion positions) |

## Game rules and classic search players
Tic-tac-toe & Connect 4 rules, perfect minimax for tic-tac-toe, and depth-limited alpha-beta (used only as a judge).

In [ ]:
%%writefile games.py
"""Game rules and classic search players shared by tictactoe.py and selfplay.py.

A state is a tuple of cells: 1 = X (moves first), -1 = O, 0 = empty.
A move is a cell index (tic-tac-toe) or a column index (Connect 4).
"""
import random
from functools import lru_cache

import numpy as np


class Game:
    rows = cols = n_actions = 0
    symbols = {1: "X", -1: "O", 0: "."}

    def initial(self):
        return (0,) * (self.rows * self.cols)

    def to_move(self, s):
        return 1 if s.count(1) == s.count(-1) else -1

    def is_over(self, s):
        return self.winner(s) != 0 or 0 not in s

    def encode(self, s):
        """One-hot from the mover's view: [my pieces | opponent pieces | empty]."""
        p = np.array(s) * self.to_move(s)
        return np.concatenate([p == 1, p == -1, p == 0]).astype(np.float32)

    def symmetries(self):
        """List of (cell_perm, action_perm): new_state[i] = state[cell_perm[i]]."""
        return self._symmetries

    def transform(self, s, cell_perm):
        return tuple(s[i] for i in cell_perm)


class TicTacToe(Game):
    name = "tictactoe"
    rows = cols = 3
    n_actions = 9
    lines = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6),
             (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]

    def __init__(self):
        grid = np.arange(9).reshape(3, 3)
        self._symmetries = []
        for g in (grid, np.fliplr(grid)):  # 4 rotations x 2 reflections
            for k in range(4):
                perm = np.rot90(g, k).ravel().tolist()
                self._symmetries.append((perm, perm))

    def legal_moves(self, s):
        return [i for i, v in enumerate(s) if v == 0]

    def play(self, s, move):
        b = list(s)
        b[move] = self.to_move(s)
        return tuple(b)

    def winner(self, s):
        for a, b, c in self.lines:
            if s[a] and s[a] == s[b] == s[c]:
                return s[a]
        return 0

    def canonical(self, s):
        """The same representative for all 8 rotations/reflections of a board."""
        return min(self.transform(s, perm) for perm, _ in self._symmetries)

    def show(self, s):
        cells = [self.symbols[v] if v else str(i + 1) for i, v in enumerate(s)]
        rows = [" | ".join(cells[r:r + 3]) for r in (0, 3, 6)]
        print("\n" + "\n---------\n".join(rows) + "\n")


class ConnectFour(Game):
    name = "connect4"
    rows, cols = 6, 7
    n_actions = 7

    def __init__(self):
        R, C = self.rows, self.cols
        self.lines = []
        for r in range(R):
            for c in range(C):
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    cells = [(r + k * dr, c + k * dc) for k in range(4)]
                    if all(0 <= rr < R and 0 <= cc < C for rr, cc in cells):
                        self.lines.append(tuple(rr * C + cc for rr, cc in cells))
        grid = np.arange(R * C).reshape(R, C)
        self._symmetries = [(grid.ravel().tolist(), list(range(C))),
                            (np.fliplr(grid).ravel().tolist(), list(range(C))[::-1])]

    def legal_moves(self, s):
        return [c for c in range(self.cols) if s[c] == 0]  # top cell still empty

    def play(self, s, col):
        b = list(s)
        for r in range(self.rows - 1, -1, -1):
            i = r * self.cols + col
            if b[i] == 0:
                b[i] = self.to_move(s)
                return tuple(b)
        raise ValueError(f"column {col} is full")

    def winner(self, s):
        for a, b, c, d in self.lines:
            if s[a] and s[a] == s[b] == s[c] == s[d]:
                return s[a]
        return 0

    def show(self, s):
        print()
        for r in range(self.rows):
            print(" ".join(self.symbols[v] for v in s[r * self.cols:(r + 1) * self.cols]))
        print(" ".join(str(c + 1) for c in range(self.cols)) + "\n")


# ---------- Perfect tic-tac-toe (minimax) ----------

TTT = TicTacToe()


@lru_cache(maxsize=None)
def ttt_score(s):
    """Perfect-play score for the player to move.
    Positive = win (faster wins score higher), 0 = draw, negative = loss."""
    if TTT.winner(s):
        return -(s.count(0) + 1)  # previous player just won
    if 0 not in s:
        return 0
    return max(-ttt_score(TTT.play(s, m)) for m in TTT.legal_moves(s))


def ttt_best_moves(s):
    scores = {m: -ttt_score(TTT.play(s, m)) for m in TTT.legal_moves(s)}
    top = max(scores.values())
    return [m for m, v in scores.items() if v == top]


# ---------- Depth-limited alpha-beta (for games too big for full minimax) ----------

def heuristic(game, s):
    """Rough score for the player to move: open lines of 2-3 pieces, center control."""
    me = game.to_move(s)
    weights = (0, 1, 4, 16, 0)
    score = 0
    for line in game.lines:
        vals = [s[i] for i in line]
        mine, theirs = vals.count(me), vals.count(-me)
        if theirs == 0:
            score += weights[mine]
        elif mine == 0:
            score -= weights[theirs]
    center = [s[r * game.cols + game.cols // 2] for r in range(game.rows)]
    return score + 3 * (center.count(me) - center.count(-me))


def alphabeta(game, s, depth, alpha=-float("inf"), beta=float("inf")):
    if game.winner(s):
        return -(10_000 + depth)  # previous player won; losing sooner is worse
    if 0 not in s:
        return 0
    if depth == 0:
        return heuristic(game, s)
    moves = sorted(game.legal_moves(s), key=lambda m: abs(m - game.cols // 2))
    for m in moves:
        alpha = max(alpha, -alphabeta(game, game.play(s, m), depth - 1, -beta, -alpha))
        if alpha >= beta:
            break
    return alpha


def alphabeta_move(game, s, depth):
    scores = {m: -alphabeta(game, game.play(s, m), depth - 1) for m in game.legal_moves(s)}
    top = max(scores.values())
    return random.choice([m for m, v in scores.items() if v == top])


# ---------- Helpers ----------

def all_positions(game):
    """Every reachable, unfinished state (only feasible for small games)."""
    seen, stack = set(), [game.initial()]
    while stack:
        s = stack.pop()
        if s in seen or game.is_over(s):
            continue
        seen.add(s)
        stack.extend(game.play(s, m) for m in game.legal_moves(s))
    return sorted(seen)


def random_player(game):
    return lambda s: random.choice(game.legal_moves(s))


def play_game(game, x_player, o_player):
    s = game.initial()
    while not game.is_over(s):
        player = x_player if game.to_move(s) == 1 else o_player
        s = game.play(s, player(s))
    return game.winner(s)


def match(game, ai, opponent, games):
    """AI alternates between moving first and second."""
    results = {"win": 0, "draw": 0, "loss": 0}
    for g in range(games):
        ai_is_x = g % 2 == 0
        w = play_game(game, ai, opponent) if ai_is_x else play_game(game, opponent, ai)
        ai_mark = 1 if ai_is_x else -1
        results["draw" if w == 0 else "win" if w == ai_mark else "loss"] += 1
    return results


def human_vs_ai(game, ai_move, explain=None):
    human = 1 if input("Play first (X)? [Y/n] ").strip().lower() != "n" else -1
    s = game.initial()
    while not game.is_over(s):
        game.show(s)
        if game.to_move(s) == human:
            legal, move = game.legal_moves(s), None
            while move not in legal:
                text = input(f"Your move (1-{game.n_actions}): ").strip()
                move = int(text) - 1 if text.isdigit() else None
        else:
            if explain:
                explain(s)
            move = ai_move(s)
            print(f"AI plays {move + 1}")
        s = game.play(s, move)
    game.show(s)
    w = game.winner(s)
    print("Draw!" if w == 0 else "You win!" if w == human else "AI wins!")

## 1, 2, 3, 5 — Supervised MLP (imitating minimax)
Two-headed network: a **policy** head (which move) and a **value** head (win/draw/loss).

In [ ]:
%%writefile tictactoe.py
"""Supervised tic-tac-toe: an MLP learns to play by imitating minimax.

Experiments:
  1. Generalization - half of the positions are held out, and the model is
     scored on positions it never saw during training.
  2. Symmetry - each board has 8 rotations/reflections with the same answer.
     Positions are split by symmetry group (so a rotated copy can't leak into
     the test set) and the training data is augmented with all 8 versions.
  3. Model size - several network sizes are compared to find the smallest one
     that still plays perfectly.
  5. Policy + value heads - the network predicts the best move AND the game
     outcome (win / draw / loss) for the player to move.

Usage:
    python tictactoe.py train   # run experiments, save the smallest perfect model
    python tictactoe.py play    # play against it
"""
import os
import sys
import random

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import tensorflow as tf

from games import (TicTacToe, all_positions, ttt_best_moves, ttt_score,
                   match, random_player, human_vs_ai)

MODEL_PATH = "tictactoe_mlp.keras"
SIZES = [(8,), (16,), (32,), (64,), (32, 32), (64, 64), (128, 128)]
GAME = TicTacToe()


# ---------- Data ----------

def targets(board):
    policy = np.zeros(9, dtype=np.float32)
    moves = ttt_best_moves(board)
    policy[moves] = 1.0 / len(moves)  # spread over all optimal moves
    value = np.float32(np.sign(ttt_score(board)))  # +1 win, 0 draw, -1 loss
    return policy, value


def make_dataset(boards, augment=True):
    syms = GAME.symmetries() if augment else GAME.symmetries()[:1]
    X, P, V = [], [], []
    for b in boards:
        policy, value = targets(b)
        for perm, _ in syms:
            X.append(GAME.encode(GAME.transform(b, perm)))
            P.append(policy[perm])
            V.append(value)
    return np.array(X), np.array(P), np.array(V)[:, None]


# ---------- Model (Functional API: one trunk, two heads) ----------

def build_model(hidden):
    inputs = tf.keras.Input(shape=(27,))
    x = inputs
    for units in hidden:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
    policy = tf.keras.layers.Dense(9, activation="softmax", name="policy")(x)
    value = tf.keras.layers.Dense(1, activation="tanh", name="value")(x)
    model = tf.keras.Model(inputs, [policy, value])
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-3),
                  loss={"policy": "categorical_crossentropy", "value": "mse"})
    return model


def fit(model, data):
    X, P, V = data
    model.fit(X, {"policy": P, "value": V}, epochs=200, batch_size=256, verbose=0,
              callbacks=[tf.keras.callbacks.ReduceLROnPlateau(
                  monitor="loss", factor=0.5, patience=15)])
    return model


def train_model(hidden, data):
    tf.keras.utils.set_random_seed(0)
    return fit(build_model(hidden), data)


def evaluate(model, boards):
    """Fraction of boards where the move is optimal / the outcome is predicted right."""
    X = np.stack([GAME.encode(b) for b in boards])
    probs, values = model.predict(X, batch_size=1024, verbose=0)
    legal = X[:, 18:] == 1  # the "empty" plane
    moves = np.argmax(np.where(legal, probs, -1), axis=1)
    move_ok = np.mean([m in ttt_best_moves(b) for m, b in zip(moves, boards)])
    true_values = np.array([np.sign(ttt_score(b)) for b in boards])
    value_ok = np.mean(np.round(values[:, 0]) == true_values)
    return move_ok, value_ok


def model_move(model, board):
    probs, _ = model(GAME.encode(board)[None], training=False)
    probs = probs.numpy()[0]
    legal = GAME.legal_moves(board)
    return legal[int(np.argmax(probs[legal]))]  # never pick an occupied cell


# ---------- Experiments ----------

def train():
    positions = all_positions(GAME)
    groups = {}
    for b in positions:
        groups.setdefault(GAME.canonical(b), []).append(b)
    unique = sorted(groups)
    random.Random(0).shuffle(unique)
    train_unique, test_unique = unique[:len(unique) // 2], unique[len(unique) // 2:]
    test_boards = [b for c in test_unique for b in groups[c]]

    print(f"{len(positions)} positions -> {len(unique)} unique up to symmetry")
    print(f"Split: train on {len(train_unique)} unique positions (x8 symmetries), "
          f"test on {len(test_boards)} never-seen positions\n")

    train_data = make_dataset(train_unique)
    all_data = make_dataset(unique)

    print(f"{'hidden':<12}{'params':>8} | {'UNSEEN move':>12}{'value':>8} | {'ALL-DATA move':>14}{'value':>8}")
    print("-" * 68)
    results = []
    for hidden in SIZES:
        held_out = train_model(hidden, train_data)
        unseen_move, unseen_value = evaluate(held_out, test_boards)
        full = train_model(hidden, all_data)
        all_move, all_value = evaluate(full, positions)
        results.append((hidden, full, all_move))
        print(f"{str(hidden):<12}{full.count_params():>8} | {unseen_move:>12.1%}{unseen_value:>8.1%} "
              f"| {all_move:>14.1%}{all_value:>8.1%}")

    no_sym = train_model((64, 64), make_dataset(train_unique, augment=False))
    move_ok, value_ok = evaluate(no_sym, test_boards)
    print(f"\nSymmetry check, (64, 64) trained WITHOUT augmentation: "
          f"unseen move {move_ok:.1%}, value {value_ok:.1%}")

    perfect = [r for r in results if r[2] == 1.0]
    hidden, model, _ = perfect[0] if perfect else max(results, key=lambda r: r[2])
    model.save(MODEL_PATH)
    print(f"\nSaved {'smallest perfect' if perfect else 'best'} model {hidden} "
          f"({model.count_params()} params) to {MODEL_PATH}")

    random.seed(0)
    ai = lambda s: model_move(model, s)
    minimax = lambda s: random.choice(ttt_best_moves(s))
    print(f"vs random  (1000 games): {match(GAME, ai, random_player(GAME), 1000)}")
    print(f"vs minimax (200 games):  {match(GAME, ai, minimax, 200)}")


def play():
    model = tf.keras.models.load_model(MODEL_PATH)

    def explain(board):
        _, value = model(GAME.encode(board)[None], training=False)
        v = float(value[0, 0])
        verdict = "I'm winning" if v > 0.5 else "I'm losing" if v < -0.5 else "it's a draw"
        print(f"AI thinks {verdict} (value {v:+.2f})")

    human_vs_ai(GAME, lambda s: model_move(model, s), explain)


if __name__ == "__main__":
    {"train": train, "play": play}.get(sys.argv[1] if len(sys.argv) > 1 else "train", train)()

In [ ]:
!python tictactoe.py train

## 6 — Self-play reinforcement learning (no teacher)
Deep Q-Learning: each move is trained toward +1 (win), 0 (draw), or −(opponent's best score).

In [ ]:
%%writefile selfplay.py
"""Self-play reinforcement learning: the AI teaches itself, with no minimax teacher.

Deep Q-Learning with an MLP. The network scores every move from the view of the
player about to move (+1 = win, 0 = draw, -1 = loss). It plays many games against
itself at once, and each move is trained toward:
    +1                          if the move wins
     0                          if the move draws
    -(opponent's best score)    otherwise (what's good for them is bad for us)

Minimax / alpha-beta are used only as judges after training.

Usage:
    python selfplay.py train tictactoe
    python selfplay.py train connect4
    python selfplay.py eval  connect4
    python selfplay.py play  connect4
"""
import os
import sys
import time
import random

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import tensorflow as tf

from games import (TicTacToe, ConnectFour, all_positions, ttt_best_moves,
                   alphabeta_move, match, random_player, human_vs_ai)

CONFIGS = {
    "tictactoe": dict(game=TicTacToe, hidden=(128, 128), envs=64, iterations=2000, updates=2),
    "connect4": dict(game=ConnectFour, hidden=(512, 256, 128), envs=128, iterations=6000, updates=4),
}
BATCH_SIZE = 512
BUFFER_SIZE = 200_000
TARGET_SYNC = 250  # gradient steps between target-network refreshes


class ReplayBuffer:
    def __init__(self, capacity, dim, n_actions):
        self.s = np.zeros((capacity, dim), np.float32)
        self.a = np.zeros(capacity, np.int32)
        self.r = np.zeros(capacity, np.float32)
        self.s2 = np.zeros((capacity, dim), np.float32)
        self.legal2 = np.zeros((capacity, n_actions), bool)
        self.done = np.zeros(capacity, bool)
        self.capacity, self.size, self.pos = capacity, 0, 0

    def add(self, s, a, r, s2, legal2, done):
        idx = (self.pos + np.arange(len(a))) % self.capacity
        self.s[idx], self.a[idx], self.r[idx] = s, a, r
        self.s2[idx], self.legal2[idx], self.done[idx] = s2, legal2, done
        self.pos = (self.pos + len(a)) % self.capacity
        self.size = min(self.size + len(a), self.capacity)

    def sample(self, n):
        i = np.random.randint(0, self.size, n)
        return self.s[i], self.a[i], self.r[i], self.s2[i], self.legal2[i], self.done[i]


def build_q_network(dim, n_actions, hidden):
    inputs = tf.keras.Input(shape=(dim,))
    x = inputs
    for units in hidden:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
    q = tf.keras.layers.Dense(n_actions, activation="tanh", name="q")(x)
    return tf.keras.Model(inputs, q)


class Agent:
    def __init__(self, game, model):
        self.game, self.model = game, model
        self.target = tf.keras.models.clone_model(model)
        self.target.set_weights(model.get_weights())
        self.optimizer = tf.keras.optimizers.Adam(1e-3)

    @tf.function(reduce_retracing=True)
    def q_values(self, x):
        return self.model(x, training=False)

    @tf.function
    def train_step(self, s, a, r, s2, legal2, done):
        # Double DQN: the online net picks the opponent's reply, the target net scores it
        next_a = tf.argmax(tf.where(legal2, self.model(s2, training=False), -2.0), axis=1)
        next_q = tf.gather(self.target(s2, training=False), next_a, batch_dims=1)
        target = tf.where(done, r, -next_q)
        with tf.GradientTape() as tape:
            q = self.model(s, training=True)
            q_a = tf.gather(q, a, batch_dims=1)
            loss = tf.reduce_mean(tf.square(target - q_a))
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def sync_target(self):
        self.target.set_weights(self.model.get_weights())

    def best_move(self, s):
        q = self.q_values(self.game.encode(s)[None]).numpy()[0]
        legal = self.game.legal_moves(s)
        return legal[int(np.argmax(q[legal]))]


# ---------- Progress checks (judges only, never used for training) ----------

class TicTacToeJudge:
    def __init__(self, game):
        self.positions = all_positions(game)
        self.X = np.stack([game.encode(s) for s in self.positions])
        self.best = [ttt_best_moves(s) for s in self.positions]

    def optimal_rate(self, agent):
        q = agent.q_values(self.X).numpy()
        moves = np.argmax(np.where(self.X[:, 18:] == 1, q, -2), axis=1)
        return np.mean([m in best for m, best in zip(moves, self.best)])

    def progress(self, agent):
        return f"optimal moves {self.optimal_rate(agent):.1%}"

    def final(self, agent):
        game = agent.game
        minimax = lambda s: random.choice(ttt_best_moves(s))
        print(f"Optimal move in {self.optimal_rate(agent):.1%} of all {len(self.positions)} positions")
        print(f"vs random  (1000 games): {match(game, agent.best_move, random_player(game), 1000)}")
        print(f"vs minimax (200 games):  {match(game, agent.best_move, minimax, 200)}")


class ConnectFourJudge:
    def __init__(self, game):
        self.game = game

    def progress(self, agent):
        r = match(self.game, agent.best_move, random_player(self.game), 40)
        return f"vs random {r['win']}/40 wins"

    def final(self, agent):
        game = agent.game
        print(f"vs random            (200 games): {match(game, agent.best_move, random_player(game), 200)}")
        for depth, games in ((1, 100), (2, 100), (4, 50)):
            opponent = lambda s, d=depth: alphabeta_move(game, s, d)
            print(f"vs alpha-beta depth {depth} ({games} games): {match(game, agent.best_move, opponent, games)}")

        positions = [self._random_position(10) for _ in range(10)]
        print("\nThinking time per move:")
        for label, player in [("MLP", agent.best_move)] + [
                (f"alpha-beta depth {d}", lambda s, d=d: alphabeta_move(game, s, d)) for d in (4, 6)]:
            start = time.perf_counter()
            for s in positions:
                player(s)
            print(f"  {label:<20} {(time.perf_counter() - start) / len(positions) * 1000:8.1f} ms")

    def _random_position(self, n_moves):
        s = self.game.initial()
        while True:
            s = self.game.initial()
            for _ in range(n_moves):
                s = self.game.play(s, random.choice(self.game.legal_moves(s)))
                if self.game.is_over(s):
                    break
            if not self.game.is_over(s):
                return s


JUDGES = {"tictactoe": TicTacToeJudge, "connect4": ConnectFourJudge}


# ---------- Training ----------

def train(name):
    cfg = CONFIGS[name]
    game = cfg["game"]()
    judge = JUDGES[name](game)
    dim = len(game.encode(game.initial()))
    tf.keras.utils.set_random_seed(0)
    agent = Agent(game, build_q_network(dim, game.n_actions, cfg["hidden"]))
    buffer = ReplayBuffer(BUFFER_SIZE, dim, game.n_actions)
    syms = [(cell_perm, np.argsort(action_perm)) for cell_perm, action_perm in game.symmetries()]

    envs = [game.initial()] * cfg["envs"]
    iterations, steps, games_played, loss = cfg["iterations"], 0, 0, 0.0
    start = time.time()
    for it in range(1, iterations + 1):
        epsilon = max(0.1, 1.0 - it / (0.6 * iterations))  # explore a lot early, less later
        q = agent.q_values(np.stack([game.encode(s) for s in envs])).numpy()
        batch = [[] for _ in range(6)]
        for i, s in enumerate(envs):
            legal = game.legal_moves(s)
            if random.random() < epsilon:
                a = random.choice(legal)
            else:
                a = legal[int(np.argmax(q[i, legal]))]
            s2 = game.play(s, a)
            won = game.winner(s2) != 0
            done = won or 0 not in s2
            for cell_perm, action_inverse in syms:  # store every symmetric version
                ts2 = game.transform(s2, cell_perm)
                legal2 = np.zeros(game.n_actions, bool)
                legal2[game.legal_moves(ts2)] = True
                for column, value in zip(batch, (game.encode(game.transform(s, cell_perm)),
                                                 action_inverse[a], float(won),
                                                 game.encode(ts2), legal2, done)):
                    column.append(value)
            if done:
                games_played += 1
            envs[i] = game.initial() if done else s2
        buffer.add(*map(np.array, batch))

        if buffer.size >= BATCH_SIZE:
            for _ in range(cfg["updates"]):
                loss = float(agent.train_step(*buffer.sample(BATCH_SIZE)))
                steps += 1
                if steps % TARGET_SYNC == 0:
                    agent.sync_target()

        if it % (iterations // 10) == 0:
            print(f"iter {it:>5}/{iterations}  games {games_played:>7}  eps {epsilon:.2f}  "
                  f"loss {loss:.4f}  {judge.progress(agent)}  [{time.time() - start:.0f}s]",
                  flush=True)

    path = f"selfplay_{name}.keras"
    agent.model.save(path)
    print(f"\nSaved {path}\n")
    random.seed(0)
    judge.final(agent)


def load_agent(name):
    game = CONFIGS[name]["game"]()
    return Agent(game, tf.keras.models.load_model(f"selfplay_{name}.keras"))


def evaluate(name):
    agent = load_agent(name)
    random.seed(0)
    JUDGES[name](agent.game).final(agent)


def play(name):
    agent = load_agent(name)

    def explain(s):
        q = agent.q_values(agent.game.encode(s)[None]).numpy()[0]
        scores = "  ".join(f"{m + 1}:{q[m]:+.2f}" for m in agent.game.legal_moves(s))
        print(f"AI move scores (+1 = win, -1 = loss): {scores}")

    human_vs_ai(agent.game, agent.best_move, explain)


if __name__ == "__main__":
    command = sys.argv[1] if len(sys.argv) > 1 else "train"
    name = sys.argv[2] if len(sys.argv) > 2 else "tictactoe"
    {"train": train, "eval": evaluate, "play": play}[command](name)

In [ ]:
!python selfplay.py train tictactoe

## 7 — Connect 4
Same self-play algorithm on a much bigger game. Evaluated against alpha-beta search at different depths, plus thinking time per move.

In [ ]:
!python selfplay.py train connect4

### Sample game: MLP (X) vs alpha-beta depth 2 (O)

In [ ]:
import random
from games import alphabeta_move
from selfplay import load_agent

agent = load_agent("connect4")
game = agent.game
random.seed(1)
s = game.initial()
while not game.is_over(s):
    if game.to_move(s) == 1:
        move, who = agent.best_move(s), "MLP (X)"
    else:
        move, who = alphabeta_move(game, s, 2), "alpha-beta depth 2 (O)"
    s = game.play(s, move)
    print(f"{who:<24} plays column {move + 1}")
game.show(s)
print({1: "MLP wins!", -1: "Alpha-beta wins!", 0: "Draw!"}[game.winner(s)])

## Play against the AIs yourself
Download the scripts (and the `.keras` models from the Output tab), then run locally:

```bash
python tictactoe.py play            # tic-tac-toe, learned from minimax
python selfplay.py play tictactoe   # tic-tac-toe, self-taught
python selfplay.py play connect4    # Connect 4, self-taught
```